# Mini-Projeto 2 - Classificação de Dígitos MNIST

Neste notebook eu montei um pipeline de análise preditiva para classificar dígitos manuscritos usando a base MNIST. A ideia é carregar os dados, entender a estrutura das imagens, treinar três modelos diferentes e comparar os resultados.

In [ ]:
import importlib
from IPython.display import Markdown, display

import mini_projeto_2 as mp
importlib.reload(mp)

mp.preparar_pastas()

## Fase 1 - Carregamento e análise exploratória

Primeiro eu carrego o MNIST e observo o tamanho da base. Também verifico se os dígitos estão bem distribuídos e mostro exemplos visuais de cada número.

In [ ]:
X, y = mp.carregar_mnist()

distribuicao_base = mp.mostrar_distribuicao_classes(
    y,
    nome_arquivo="distribuicao_classes_base_completa.png"
)

mp.mostrar_grade_digitos(X, y)

Cada imagem do MNIST tem 28 por 28 pixels. No modelo, essa imagem é representada como uma linha com 784 colunas, porque 28 x 28 = 784. Cada coluna guarda a intensidade de um pixel, indo de 0 para regiões escuras até 255 para regiões mais claras.

## Amostra para testes no VS Code

Para facilitar os testes, deixei uma amostra rápida ativada no arquivo Python. Assim o projeto roda mais rápido enquanto estou desenvolvendo. Se eu quiser usar a base completa, basta trocar `USAR_AMOSTRA_RAPIDA` para `False`.

In [ ]:
if mp.USAR_AMOSTRA_RAPIDA:
    X_modelo, y_modelo = mp.criar_amostra_estratificada(
        X,
        y,
        n_por_classe=mp.AMOSTRA_POR_CLASSE
    )
else:
    X_modelo, y_modelo = X, y

distribuicao_modelagem = mp.mostrar_distribuicao_classes(
    y_modelo,
    nome_arquivo="distribuicao_classes_modelagem.png"
)

## Fase 2 - Pré-processamento e divisão dos dados

Aqui eu separo a base em treino, validação e teste usando estratificação. Depois faço a normalização dos pixels dividindo por 255, deixando os valores entre 0 e 1.

In [ ]:
X_treino, X_validacao, X_teste, y_treino, y_validacao, y_teste = mp.dividir_e_normalizar(
    X_modelo,
    y_modelo
)

A normalização é importante porque os modelos trabalham melhor quando os valores ficam em uma escala parecida. Como os pixels iam de 0 a 255, dividir por 255 deixa tudo no intervalo de 0 até 1.

## Fase 3 - Treinamento dos três modelos

Nesta etapa eu treino KNN, Random Forest e MLP. Para cada modelo, testo configurações diferentes e escolho a melhor usando o conjunto de validação.

In [ ]:
melhores_modelos, historico_df = mp.treinar_modelos(
    X_treino,
    y_treino,
    X_validacao,
    y_validacao
)

historico_df.round(2)

Eu usei o conjunto de validação para comparar as configurações antes de olhar o teste final. Isso ajuda a evitar escolher um modelo só porque ele foi bem em um conjunto que deveria ficar reservado para a avaliação final.

## Fase 4 - Avaliação comparativa

Agora eu avalio os melhores modelos no conjunto de teste. As métricas aparecem em porcentagem e as matrizes de confusão mostram onde cada modelo acertou ou errou.

In [ ]:
comparativo, predicoes = mp.avaliar_modelos(melhores_modelos, X_teste, y_teste)
comparativo.round(2)

In [ ]:
melhor_nome = comparativo.iloc[0]["modelo"]
digito_real, digito_previsto, total = mp.identificar_maior_confusao(
    y_teste,
    predicoes[melhor_nome]
)

display(Markdown(
    f"O melhor modelo foi **{melhor_nome}**. "
    f"A maior confusão apareceu quando o dígito **{digito_real}** foi previsto como **{digito_previsto}**, "
    f"com **{total}** casos. Isso mostra que alguns dígitos têm formatos parecidos e podem gerar erro mesmo quando o resultado geral é bom."
))

## Fase 5 - Robustez com classes ocultas

Nesta parte eu escondo os dígitos 4 e 7 do treino. Depois testo o modelo somente com essas classes para ver como ele reage quando recebe imagens que nunca viu durante o aprendizado.

In [ ]:
resultado_ood = mp.treinar_teste_ood(
    X_treino,
    y_treino,
    X_teste,
    y_teste,
    melhores_modelos,
    classes_ocultas=(4, 7)
)

In [ ]:
texto_ood = (
    "Quando o modelo recebe dígitos que não apareceram no treino, ele ainda tenta encaixar a imagem em alguma classe conhecida. "
    "Isso mostra a falsa certeza: o modelo pode responder com confiança mesmo quando não aprendeu aquela classe."
)

if resultado_ood["certeza_media_%"] is not None:
    texto_ood += f" Neste teste, a certeza média foi de {resultado_ood['certeza_media_%']:.2f}%."

display(Markdown(texto_ood))

## Fase 5.3 - Teste com imagens próprias

Para testar uma imagem minha, eu posso colocar arquivos na pasta `data/minhas_imagens`. O código converte a imagem para escala de cinza, ajusta para 28x28 pixels e usa o melhor modelo para fazer a previsão.

In [ ]:
caminho_modelo = mp.salvar_melhor_modelo(melhores_modelos, comparativo)
previsoes_imagens_proprias = mp.testar_imagens_proprias(
    melhores_modelos[melhor_nome]["modelo"]
)

previsoes_imagens_proprias

## Conclusão

O modelo escolhido deve ser o que apresentar melhor equilíbrio entre acurácia no teste e tempo de execução. Neste projeto, eu comparo os três modelos com as mesmas métricas para justificar a escolha final, em vez de olhar apenas um resultado isolado.